In [1]:
import pandas as pd

In [2]:
laps = pd.DataFrame({
    'lap': range(1, 6),
    'time': [68.5, 58.6, 61.2, 55.9, 59.4]
    })

laps

,lap,time
0,1,68.5
1,2,58.6
2,3,61.2
3,4,55.9
4,5,59.4


### Дослідження
- Імпортуємо бібліотеку `pandas`, яка вміє працювати з таблицями, та створюємо таблицю `laps` з двома стовпцями:
  - `lap` — номер кола;
  - `time` — час проходження кола в секундах.
- У масиві часів є як повільні кола (більше 60 секунд), так і швидкі (менше 60 секунд),
  щоб можна було відфільтрувати лише найкращі спроби.


In [3]:
laps['completed'] = laps['time'].where(laps['time'] < 60.)
laps

,lap,time,completed
0,1,68.5,NaN
1,2,58.6,58.6
2,3,61.2,NaN
3,4,55.9,55.9
4,5,59.4,59.4


### Дослідження
- Вираз `laps['time'].where(laps['time'] < 60.)` залишає значення часу лише там, де коло пройдено швидше за 60 секунд.
- Для повільніших кіл (час ≥ 60 секунд) у новому стовпці `completed` з'являються значення `NaN` —
  це означає, що ці кола **не потрапили** до нашої "книги рекордів".
- Таким чином, у таблиці `laps` ми одночасно бачимо і вихідний час, і те, чи увійшло коло до списку найшвидших.


In [4]:
# Рахуємо, скільки кіл не потрапили до "книги рекордів" (time >= 60 секунд)
missing_laps = laps['completed'].isnull().sum()
print('Кола, які не пройшли за 60 секунд:', int(missing_laps))

# Формуємо окрему таблицю лише з найшвидшими колами
laps_completed = laps.dropna(subset=['completed'])
laps_completed


Кола, які не пройшли за 60 секунд: 2


,lap,time,completed
1,2,58.6,58.6
3,4,55.9,55.9
4,5,59.4,59.4


### Дослідження
- Метод `isnull().sum()` підраховує, скільки значень у стовпці `completed` є пропущеними (`NaN`).
  Це і є кількість кіл, які **не вклалися в 60 секунд**.
- Далі за допомогою `dropna(subset=['completed'])`, яка видаляє рядки, де в `completed` стоїть `NaN`, ми формуємо нову таблицю `laps_completed`,
  де залишаються тільки ті рядки, де значення `completed` не є пропущеним — тобто лише найшвидші кола.


In [5]:
# Стандартизуємо час найшвидших кіл
mean_time = laps_completed['completed'].mean()
std_time = laps_completed['completed'].std()

laps_completed['standard'] = (laps_completed['completed'] - mean_time) / std_time
laps_completed


C:\Users\User\AppData\Local\Temp\ipykernel_580\663437873.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laps_completed['standard'] = (laps_completed['completed'] - mean_time) / std_time


,lap,time,completed,standard
1,2,58.6,58.6,0.34534
3,4,55.9,55.9,-1.12690
4,5,59.4,59.4,0.78156


### Дослідження
- Обчислюємо **середній час** найшвидших кіл (`mean_time`) та їх **стандартне відхилення** (`std_time`) за допомогою методу `std()`.
- `mean_time` — це середній час серед усіх найшвидших кіл.
- `std_time` — стандартне відхилення, яке показує, наскільки сильно часи розкидані навколо середнього.
- Нова колонка `standard` показує **стандартизований час** кожного кола:
  - віднімаємо середнє `mean_time`,
  - ділимо на стандартне відхилення `std_time`.
- Що це дає:
  - значення **близькі до 0** — це кола з «середнім» часом,
  - **додатні** значення — кола повільніші за середній час,
  - **від’ємні** значення — кола, які пройдені швидше за середній час.
- Таке представлення зручно для подальшого аналізу та порівняння різних перегонів.
